In [ ]:
import os
import sys
import pandas as pd
import django

# ==========================================================
# 🚀 STEP 1: SET THE PROJECT ROOT
# ==========================================================
project_root = r"C:\Bhumala\Real Estate build 28-apr-2026\Real_Estate_Platform 24-04-26\Real_Estate_Platform"

if project_root not in sys.path:
    sys.path.append(project_root)

os.chdir(project_root)

# ==========================================================
# 🚀 STEP 2: SETUP DJANGO & ALLOW ASYNC
# ==========================================================
os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'Real_Estate.settings')

# ❗ THIS IS THE FIX FOR THE "ASYNC CONTEXT" ERROR
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"

try:
    django.setup()
    print("✅ Django Setup Successful!")
except Exception as e:
    print(f"❌ Django Setup Failed: {e}")

# ==========================================================
# 🚀 STEP 3: IMPORT MODELS
# ==========================================================
try:
    from Admin_App.models import (
        RentalResidentialProperty, CommercialRentalProperty, PGColivingProperty,
        ResaleResidentialProperty, CommercialResaleProperty, PlotSaleProperty,
        AgriculturalResaleProperty, IndustrialResaleProperty
    )
    print("✅ Models Imported Successfully!")
except Exception as e:
    print(f"❌ Model Import Failed: {e}")

# ==========================================================
# 🚀 STEP 4: SYNC DATABASE TO CSV
# ==========================================================
def sync_database_to_csv():
    models_to_sync = [
        (RentalResidentialProperty, "Residential Data"),
        (CommercialRentalProperty, "Commercial Data"),
        (PGColivingProperty, "PG Data"),
        (ResaleResidentialProperty, "Resale Residential"),
        (CommercialResaleProperty, "Commercial Resale"),
        (PlotSaleProperty, "Plot Resale"),
        (AgriculturalResaleProperty, "Agricultural Data"),
        (IndustrialResaleProperty, "Industrial Resale"),
    ]
    
    all_records = []
    for model_class, source_name in models_to_sync:
        try:
            # Now that ASYNC_UNSAFE is true, this will work
            db_data = list(model_class.objects.all().values())
            for item in db_data:
                item['db_id'] = item['id']
                item['source_sheet'] = source_name
                all_records.append(item)
            print(f"📍 Loaded {len(db_data)} items from {source_name}")
        except Exception as e:
            print(f"⚠️ Error reading {source_name}: {e}")

    if not all_records:
        print("❌ ERROR: No real data found in Database! Please add properties in Django Admin first.")
        return

    # SAVE TO Main_App/ml_models/df.csv
    df = pd.DataFrame(all_records)
    output_path = os.path.join(project_root, "Main_App", "ml_models", "df.csv")
    df.to_csv(output_path, index=False)
    print(f"\n🚀 SUCCESS! New df.csv created at: {output_path}")

sync_database_to_csv()




✅ Django Setup Successful!
✅ Models Imported Successfully!
📍 Loaded 20 items from Residential Data
📍 Loaded 3 items from Commercial Data
📍 Loaded 9 items from PG Data
📍 Loaded 1 items from Resale Residential
📍 Loaded 0 items from Commercial Resale
📍 Loaded 0 items from Plot Resale
📍 Loaded 0 items from Agricultural Data
📍 Loaded 0 items from Industrial Resale

🚀 SUCCESS! New df.csv created at: C:\Bhumala\Real Estate build 28-apr-2026\Real_Estate_Platform 24-04-26\Real_Estate_Platform\Main_App\ml_models\df.csv


In [ ]:
import os
import pandas as pd
import numpy as np
import faiss
import pickle
from sentence_transformers import SentenceTransformer

# 1. SET PATHS
# Use the same project_root from your previous successful code
project_root = r"C:\Bhumala\Real Estate build 28-apr-2026\Real_Estate_Platform 24-04-26\Real_Estate_Platform"
csv_path = os.path.join(project_root, "Main_App", "ml_models", "df.csv")
index_path = os.path.join(project_root, "Main_App", "ml_models", "faiss_index.bin")

# 2. LOAD THE NEW CSV
print("📖 Loading new df.csv...")
df = pd.read_csv(csv_path)

# 3. PREPARE TEXT FOR AI
# We combine the title/description so the AI can 'read' it
def prepare_text(row):
    return f"{row.get('bhk_type', '')} {row.get('source_sheet', '')} in {row.get('locality', '')} {row.get('city', '')}"

df['combined_text'] = df.apply(prepare_text, axis=1)
sentences = df['combined_text'].fillna("property").tolist()

# 4. CREATE EMBEDDINGS (THE 'VECTORS')
print("🧠 Generating AI embeddings for 33 properties...")
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(sentences).astype('float32')

# 5. SAVE NEW FAISS INDEX
print("💾 Saving new faiss_index.bin...")
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

faiss.write_index(index, index_path)

print(f"\n🚀 ALL DONE! AI Index updated with {index.ntotal} real properties.")
print("Now RESTART your Django server and search for one of your properties.")

📖 Loading new df.csv...
🧠 Generating AI embeddings for 33 properties...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

💾 Saving new faiss_index.bin...

🚀 ALL DONE! AI Index updated with 33 real properties.
Now RESTART your Django server and search for one of your properties.
